In [30]:
import numpy as np

In [31]:
x = np.array([1, 2])
w1 = np.array([[1, 2], [-1, 1]])
b1 = np.zeros(2)
w2 = np.array([2, 1])
b2 = 1.0

In [32]:
def forward(x):
    z1 = x @ w1 + b1
    h = np.maximum(0, z1)
    return z1, h, z1 @ w2 + b2  # z1 | h


z1, h, y_hat = forward(x)
print(z1, h, y_hat)

[-1.  4.] [0. 4.] 3.0


In [33]:
def mse(y, p):
    return np.mean((p - y) ** 2)


def bce(y, p):
    return np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


print(mse(np.array([2]), np.array([5])))

for p in [0.9, 0.5, 0.1]:
    print(p, round(bce(1, p), 4))

9.0
0.9 -0.1054
0.5 -0.6931
0.1 -2.3026


In [34]:
print(mse(np.array([8]), np.array([6])))

4.0


In [35]:
print(mse(np.array([8]), np.array([5])))
print(mse(np.array([4]), np.array([4])))
print(mse(np.array([6]), np.array([9])))

9.0
0.0
9.0


In [36]:
print(round(bce(1, 0.8), 2))
print(round(bce(0, 0.8), 2))

-0.22
-1.61


In [37]:
print(round(bce(1, 0.01), 2))

-4.61


In [38]:
def backword(x, y):
    z1, h, y_hat = forward(x)
    d = 2 * (y_hat - y)
    dw2 = h * d
    db = w2 * d
    dz1 = db * (z1 > 0)
    dw1 = np.outer(x, dz1)
    return dw1, dz1, dw2, d

In [39]:
y = 2.0
dW1, db1, dw2, db2 = backword(x, y)
print(dW1)
print(db1)
print(dw2, db2)

[[0. 2.]
 [0. 4.]]
[0. 2.]
[0. 8.] 2.0


In [40]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))


class MLP:
    def __init__(self, n_in, n_h, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.normal(0, 1, (n_in, n_h))
        self.W2 = r.normal(0, 1, (n_h, 1))
        self.b1, self.b2 = np.zeros(n_h), np.zeros(1)

    def forward(self, X):
        self.h = sigmoid(X @ self.W1 + self.b1)
        self.p = sigmoid(self.h @ self.W2 + self.b2)
        return self.p

    def step(self, X, y, lr):
        dz2 = (self.p - y) / len(X)
        dz1 = dz2 @ self.W2.T * self.h * (1 - self.h)
        self.W2 = lr * self.h.T @ dz2
        self.b2 = lr * dz2.sum(0)
        self.W1 = lr * X.T @ dz1
        self.b1 = lr * dz1.sum(0)

In [41]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)
net = MLP(2, 4, seed=1)

for ep in range(5001):
    p = net.forward(X)
    loss = np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    if ep in (0, 500, 1000, 2000, 5000):
        print(ep, round(loss, 4))

    net.step(X, y, lr=0.1)


print((net.forward(X) > 0.5).astype(int).ravel())

0 -0.7452
500 -0.6931
1000 -0.6931
2000 -0.6931
5000 -0.6931
[0 0 0 0]


In [45]:
import torch

x = torch.tensor([1.0, 2.0])
W1 = torch.tensor([[1.0, 2.0], [-1.0, 1.0]], requires_grad=True)
b1 = torch.zeros(2, requires_grad=True)
w2 = torch.tensor([2.0, 1.0], requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)

y_hat = torch.relu(x @ W1 + b1) @ w2 + b2
loss = (y_hat - 2) ** 2
loss.backward()

print(W1.grad.numpy())
print(b1.grad.numpy())
print(w2.grad.numpy(), b2.grad.item())

[[ 0.  6.]
 [ 0. 12.]]
[0. 6.]
[ 0. 24.] 6.0


In [46]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

belgilar = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=belgilar)

X = df[belgilar].values
y = df["species"].astype("category").cat.codes.values
print(df.shape, np.bincount(y))

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(len(Xtr), len(Xte))

(342, 7) [151  68 123]
239 103


In [56]:
import torch, torch.nn as nn


def fit_eval(Xtr, ytr, Xte, yte, scale=True, lr=0.1, hid=8, log=False):
    # if scale:
    #     sc = StandardScaler().fit(Xtr)
    #     Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

    Xtr = torch.tensor(Xtr).float()
    Xte = torch.tensor(Xte).float()
    ytr = torch.tensor(ytr).long()
    yte = torch.tensor(yte).long()
    torch.manual_seed(42)
    model = nn.Sequential(nn.Linear(4, hid), nn.ReLU(), nn.Linear(hid, 3))

    opt = torch.optim.SGD(model.parameters(), lr=lr)

    for ep in range(201):
        loss = nn.functional.cross_entropy(model(Xtr), ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if log and ep % 50 == 0:
            print(ep, round(loss.item(), 4))
    acc = (model(Xte).argmax(1) == yte).float().mean()
    return round(acc.item(), 4)


print(fit_eval(Xtr, ytr, Xte, yte, log=True))

0 113.9657
50 1.0491
100 1.0483
150 1.0482
200 1.0482
0.4369
